# LeetCode 3396 — Minimum Number of Operations to Make Elements in Array Distinct bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-operations-to-make-elements-in-array-distinct/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table (`set`), Duyệt ngược

Skeleton LeetCode:

```python
class Solution:
    def minimumOperations(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và ví dụ được dịch từ đề gốc (Weekly Contest 429); mọi lời giải trong notebook đã được chạy và kiểm chứng bằng code thật. Trong notebook ta viết `list[int]` thay cho `List[int]` để không cần `import`; khi nộp LeetCode cả hai đều chạy được.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi thao tác **xóa 3 phần tử đầu mảng** (nếu còn ít hơn 3 thì xóa hết); cần **số thao tác ít nhất** để các phần tử còn lại **đôi một khác nhau** (mảng rỗng được coi là khác nhau).
2. Nắm các công cụ Python: **lát cắt** `nums[3:]`, **`set`** (`add`, `in`, `len`), so sánh `len(set(a)) == len(a)` để kiểm tra "mọi phần tử khác nhau", duyệt **ngược** bằng `range(n - 1, -1, -1)`, và **chia làm tròn lên** `(x + 2) // 3`.
3. Đi từ cách **mô phỏng từng thao tác** → **thử từng số thao tác `k`** → **duyệt từ phải sang trái với `set`** → **đếm tần suất bằng `Counter`/`dict`** → **tìm vị trí trùng xa nhất bên phải** rồi tính bằng công thức.
4. Hiểu mấu chốt: ta chỉ xóa được **tiền tố** (đoạn đầu), nên phần còn lại là một **hậu tố**; hậu tố dài nhất không trùng lặp bắt đầu tại một vị trí `s` cố định, và đáp án là `ceil(s / 3)`.
5. Tránh các lỗi kinh điển: dùng phép chia **làm tròn xuống** thay vì lên, đếm số phần tử trùng thay vì **vị trí** trùng, duyệt từ trái sang phải rồi dừng ở trùng lặp **đầu tiên**, và quên trường hợp còn ít hơn 3 phần tử hoặc mảng đã khác nhau sẵn.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Bạn cần làm cho các phần tử trong mảng **đôi một khác nhau** (distinct). Bạn có thể thực hiện thao tác sau **bất kỳ số lần nào**:

- Xóa **3 phần tử đầu tiên** của mảng. Nếu mảng có **ít hơn 3 phần tử**, xóa **tất cả** phần tử còn lại.

Lưu ý: mảng **rỗng** được coi là có các phần tử khác nhau. Hãy trả về **số thao tác tối thiểu** cần thực hiện.

**Ví dụ 1**

- Input: `nums = [1,2,3,4,2,3,3,5,7]`
- Output: `2`
- Giải thích: thao tác 1 xóa `[1,2,3]` còn `[4,2,3,3,5,7]`; thao tác 2 xóa `[4,2,3]` còn `[3,5,7]` — đã khác nhau.

**Ví dụ 2**

- Input: `nums = [4,5,6,4,4]`
- Output: `2`
- Giải thích: thao tác 1 xóa `[4,5,6]` còn `[4,4]`; thao tác 2 xóa nốt `[4,4]` (ít hơn 3 phần tử nên xóa hết) còn mảng rỗng.

**Ví dụ 3**

- Input: `nums = [6,7,8,9]`
- Output: `0`
- Giải thích: mảng đã gồm các phần tử khác nhau nên không cần thao tác nào.

**Ràng buộc:** `1 <= nums.length <= 100`; `1 <= nums[i] <= 100`.

In [1]:
import random
from collections import Counter

examples = [
    ([1, 2, 3, 4, 2, 3, 3, 5, 7], 2),
    ([4, 5, 6, 4, 4], 2),
    ([6, 7, 8, 9], 0),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[1, 2, 3, 4, 2, 3, 3, 5, 7] -> kỳ vọng 2
[4, 5, 6, 4, 4] -> kỳ vọng 2
[6, 7, 8, 9] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Lát cắt `nums[3:]` — "bỏ 3 phần tử đầu"

`nums[3:]` là mảng mới gồm các phần tử từ chỉ số `3` đến hết, tức là **xóa 3 phần tử đầu**. Nếu mảng có ít hơn 3 phần tử thì `nums[3:]` là `[]` (**không báo lỗi**). Đây chính xác là quy tắc "ít hơn 3 thì xóa hết" của đề.

In [2]:
nums = [1, 2, 3, 4, 5]
print(nums[3:])          # [4, 5]  (bỏ 3 phần tử đầu)
print(nums[3:][3:])      # []      (còn 2 phần tử, bỏ 3 -> rỗng, không lỗi)
print([7, 8][3:])        # []
print([][3:])            # []

[4, 5]
[]
[]
[]


### 3.2. `set` — tập hợp không có phần tử trùng

`set` là cấu trúc lưu **các giá trị khác nhau**. Thêm một giá trị đã có thì không thay đổi gì. `x in s` kiểm tra có mặt hay không (nhanh, trung bình `O(1)`). `len(s)` cho số giá trị khác nhau.

In [3]:
s = set()
s.add(3)
s.add(5)
s.add(3)                 # đã có, không thêm lần nữa
print(s, len(s))         # {3, 5} 2
print(3 in s, 4 in s)    # True False
print(set([1, 2, 2, 3, 3, 3]))   # {1, 2, 3}

{3, 5} 2
True False
{1, 2, 3}


### 3.3. Kiểm tra "mọi phần tử khác nhau"

Mảng `a` gồm các phần tử đôi một khác nhau **khi và chỉ khi** `len(set(a)) == len(a)` (loại trùng mà độ dài không đổi nghĩa là chưa có trùng). Với mảng rỗng: `len(set([])) == len([]) == 0` nên `True` — khớp với quy ước của đề.

In [4]:
def all_distinct(a):
    return len(set(a)) == len(a)


print(all_distinct([6, 7, 8, 9]))        # True
print(all_distinct([4, 4]))               # False
print(all_distinct([]))                   # True (mảng rỗng được coi là khác nhau)

True
False
True


### 3.4. Duyệt ngược: `range(n - 1, -1, -1)`

`range(start, stop, step)` với `step = -1` đếm **lùi**. `range(n - 1, -1, -1)` cho `n-1, n-2, ..., 0` (điểm dừng `-1` là **không gồm**, nên dừng ở `0`). Có thể viết bằng `reversed(range(n))`.

In [5]:
nums = [10, 20, 30, 40]
n = len(nums)
for i in range(n - 1, -1, -1):
    print(i, nums[i])
print(list(reversed(range(n))))          # [3, 2, 1, 0]

3 40
2 30
1 20
0 10
[3, 2, 1, 0]


### 3.5. Chia làm tròn lên (ceiling) với số nguyên

Cần xóa một đoạn đầu dài `L` phần tử, mỗi thao tác xóa tối đa `3`. Số thao tác nhỏ nhất là `ceil(L / 3)`. Với số nguyên không dùng `math.ceil(L / 3)` (số thực), mà dùng mẹo: **`(L + 2) // 3`** (cộng `3 - 1` trước khi chia nguyên). Ví dụ `L = 4`: cần `2` thao tác (`3 + 1`); `(4 + 2) // 3 = 2`. Còn `L // 3 = 1` là **sai** vì làm tròn xuống.

In [6]:
for L in range(0, 10):
    print(f"L={L}: (L+2)//3 = {(L + 2) // 3}, L//3 = {L // 3}")

L=0: (L+2)//3 = 0, L//3 = 0
L=1: (L+2)//3 = 1, L//3 = 0
L=2: (L+2)//3 = 1, L//3 = 0
L=3: (L+2)//3 = 1, L//3 = 1
L=4: (L+2)//3 = 2, L//3 = 1
L=5: (L+2)//3 = 2, L//3 = 1
L=6: (L+2)//3 = 2, L//3 = 2
L=7: (L+2)//3 = 3, L//3 = 2
L=8: (L+2)//3 = 3, L//3 = 2
L=9: (L+2)//3 = 3, L//3 = 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ta chỉ xóa được **từ đầu mảng**, mỗi lần `3` phần tử. Sau `k` thao tác, mảng còn lại là **hậu tố** `nums[3k:]`. Vậy bài toán thành: tìm `k` nhỏ nhất sao cho `nums[3k:]` gồm các phần tử khác nhau.

Với `nums = [1,2,3,4,2,3,3,5,7]` (`n = 9`):

| `k` | mảng còn lại `nums[3k:]` | khác nhau? |
|---|---|---|
| 0 | `[1,2,3,4,2,3,3,5,7]` | không (2 và 3 lặp) |
| 1 | `[4,2,3,3,5,7]` | không (3 lặp) |
| 2 | `[3,5,7]` | **có** |

Đáp án `k = 2`. Hãy thử tay `[4,5,6,4,4]`: `k=0` có `4` lặp; `k=1` còn `[4,4]` lặp; `k=2` còn `[]` rỗng ⇒ đáp án `2`. Để ý rằng khi `k` đủ lớn (`3k >= n`) mảng rỗng, luôn thỏa, nên **đáp án luôn tồn tại và `<= ceil(n / 3)`**.

In [7]:
nums = [1, 2, 3, 4, 2, 3, 3, 5, 7]
for k in range(0, 4):
    rest = nums[3 * k:]
    print(f"k={k}: còn {rest} | khác nhau = {len(set(rest)) == len(rest)}")

k=0: còn [1, 2, 3, 4, 2, 3, 3, 5, 7] | khác nhau = False
k=1: còn [4, 2, 3, 3, 5, 7] | khác nhau = False
k=2: còn [3, 5, 7] | khác nhau = True
k=3: còn [] | khác nhau = True


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng thao tác

**Ý tưởng.** Làm đúng như đề: lặp lại "nếu mảng chưa khác nhau thì xóa `3` phần tử đầu và đếm `1` thao tác". Kiểm tra khác nhau bằng `len(set(arr)) == len(arr)`.

In [8]:
def min_ops_simulate(nums: list[int]) -> int:
    arr = list(nums)                        # sao chép để không làm hỏng input
    ops = 0
    while len(set(arr)) != len(arr):        # còn phần tử trùng
        arr = arr[3:]                       # xóa 3 phần tử đầu (hoặc hết nếu ít hơn 3)
        ops += 1
    return ops


for nums, expected in examples:
    print(min_ops_simulate(nums), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`. Có tối đa `ceil(n/3)` vòng lặp; mỗi vòng tạo `set` và cắt list tốn `O(n)` ⇒ **Time `O(n²)`** (chính xác hơn `O(n · n/3)`), **Space `O(n)`**.

**Nút thắt:** mỗi vòng ta dựng lại `set` từ đầu và cắt list mới — lặp lại gần như cùng một công việc. Ta nên tìm **trực tiếp** hậu tố dài nhất không trùng lặp, thay vì thử dần.

## 6. Ý tưởng thuật toán / bước đột phá

Mảng còn lại luôn là hậu tố `nums[s:]` với `s` là bội của `3` (hoặc rỗng). Hậu tố `nums[s:]` khác nhau khi mọi phần tử trong đó khác nhau. Tính chất quan trọng: **nếu hậu tố bắt đầu tại `s` đã khác nhau thì hậu tố bắt đầu tại `s' > s` cũng khác nhau** (bớt phần tử không tạo trùng). Vì vậy ta chỉ cần tìm **hậu tố dài nhất không trùng** bắt đầu tại `p` (vị trí nhỏ nhất có `nums[p:]` khác nhau), rồi làm tròn `p` lên bội của `3`: đáp án `= ceil(p / 3)`.

**Cách tìm `p` bằng duyệt từ phải sang trái với `set`:** đi `i = n-1, n-2, ...`; thêm `nums[i]` vào `seen`; **gặp phần tử đã có trong `seen` đầu tiên** ở chỉ số `i` ⇒ hậu tố `nums[i:]` bị trùng, còn `nums[i+1:]` thì khác nhau. Do đó `p = i + 1` và đáp án `= ceil((i + 1) / 3) = (i + 3) // 3 = i // 3 + 1`. Nếu duyệt hết mà không trùng ⇒ đáp án `0`.

In [9]:
nums = [1, 2, 3, 4, 2, 3, 3, 5, 7]
seen = set()
for i in range(len(nums) - 1, -1, -1):
    x = nums[i]
    if x in seen:
        print(f"i={i}: {x} đã có trong seen={sorted(seen)} -> trùng! cần xóa tiền tố dài {i + 1} -> ops = {(i + 1 + 2) // 3}")
        break
    seen.add(x)
    print(f"i={i}: thêm {x}, seen={sorted(seen)}")

i=8: thêm 7, seen=[7]
i=7: thêm 5, seen=[5, 7]
i=6: thêm 3, seen=[3, 5, 7]
i=5: 3 đã có trong seen=[3, 5, 7] -> trùng! cần xóa tiền tố dài 6 -> ops = 2


## 7. Các cách tiếp theo

### Cách 2 — Thử từng số thao tác `k` (kiểm tra hậu tố)

**Giải thích từng đoạn.**

1. Với `k = 0, 1, 2, ...` lấy hậu tố `rest = nums[3*k:]`.
2. Nếu `len(set(rest)) == len(rest)` thì `k` là đáp án (trả về ngay, vì ta thử `k` tăng dần).
3. Vòng lặp luôn dừng: đến `3k >= n` thì `rest = []` thỏa.

Khác Cách 1 ở chỗ không sao chép/cắt dần mà tính hậu tố trực tiếp theo `k`.

In [10]:
def min_ops_try_k(nums: list[int]) -> int:
    k = 0
    while True:
        rest = nums[3 * k:]
        if len(set(rest)) == len(rest):
            return k
        k += 1


for nums, expected in examples:
    print(min_ops_try_k(nums), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionTryK:
    def minimumOperations(self, nums: list[int]) -> int:
        k = 0
        while True:
            rest = nums[3 * k:]
            if len(set(rest)) == len(rest):
                return k
            k += 1


print(SolutionTryK().minimumOperations([1, 2, 3, 4, 2, 3, 3, 5, 7]))      # 2

2


**Độ phức tạp Cách 2:** có tối đa `ceil(n/3) + 1` lần thử, mỗi lần `O(n)` ⇒ **Time `O(n²)`**, **Space `O(n)`**.

### Cách 3 — Duyệt từ phải sang trái với `set` (cách tối ưu)

**Giải thích từng đoạn.**

1. `seen = set()`: các giá trị đã gặp ở bên **phải**.
2. `for i in range(n - 1, -1, -1)`: đi từ cuối về đầu.
3. Nếu `nums[i] in seen` ⇒ hậu tố `nums[i:]` có trùng ⇒ phải xóa ít nhất `i + 1` phần tử đầu ⇒ trả về `i // 3 + 1`.
4. Ngược lại `seen.add(nums[i])`.
5. Duyệt hết không trùng ⇒ trả về `0`.

In [12]:
def min_ops_right_to_left(nums: list[int]) -> int:
    seen = set()
    for i in range(len(nums) - 1, -1, -1):
        if nums[i] in seen:                  # nums[i:] có trùng; nums[i+1:] thì không
            return i // 3 + 1                # ceil((i + 1) / 3)
        seen.add(nums[i])
    return 0


for nums, expected in examples:
    print(min_ops_right_to_left(nums), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


In [13]:
# Code nộp LeetCode cho Cách 3
class SolutionRightToLeft:
    def minimumOperations(self, nums: list[int]) -> int:
        seen = set()
        for i in range(len(nums) - 1, -1, -1):
            if nums[i] in seen:
                return i // 3 + 1
            seen.add(nums[i])
        return 0


print(SolutionRightToLeft().minimumOperations([4, 5, 6, 4, 4]))           # 2

2


**Độ phức tạp Cách 3:** **Time `O(n)`** (mỗi phần tử một lần), **Space `O(n)`** cho `seen` (tối đa `min(n, 100)` giá trị).

### Cách 4 — Đếm tần suất bằng `Counter`, vừa xóa vừa cập nhật

**Giải thích.** Đếm số lần xuất hiện mỗi giá trị bằng `Counter`. Duy trì biến `dup` = **số giá trị còn xuất hiện từ 2 lần trở lên**. Mỗi thao tác xóa 3 phần tử đầu: với mỗi phần tử bị xóa, giảm tần suất; nếu tần suất từ `2` xuống `1` thì `dup -= 1`. Dừng khi `dup == 0`. Cách này giống mô phỏng nhưng **không dựng lại `set`**, tổng công việc `O(n)`.

In [14]:
def min_ops_counter(nums: list[int]) -> int:
    cnt = Counter(nums)
    dup = sum(1 for v in cnt.values() if v >= 2)     # số giá trị còn bị trùng
    ops = 0
    pos = 0                                           # vị trí đầu mảng hiện tại
    while dup > 0:
        for _ in range(3):                            # xóa tối đa 3 phần tử đầu
            if pos >= len(nums):
                break
            x = nums[pos]
            pos += 1
            cnt[x] -= 1
            if cnt[x] == 1:                           # từ 2 xuống 1: hết trùng
                dup -= 1
        ops += 1
    return ops


for nums, expected in examples:
    print(min_ops_counter(nums), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


In [15]:
# Code nộp LeetCode cho Cách 4
class SolutionCounter:
    def minimumOperations(self, nums: list[int]) -> int:
        cnt = Counter(nums)
        dup = sum(1 for v in cnt.values() if v >= 2)
        ops = pos = 0
        while dup > 0:
            for _ in range(3):
                if pos >= len(nums):
                    break
                x = nums[pos]
                pos += 1
                cnt[x] -= 1
                if cnt[x] == 1:
                    dup -= 1
            ops += 1
        return ops


print(SolutionCounter().minimumOperations([6, 7, 8, 9]))                 # 0

0


**Độ phức tạp Cách 4:** **Time `O(n)`** (mỗi phần tử bị xóa nhiều nhất một lần), **Space `O(n)`** cho bộ đếm.

### Cách 5 — Tìm vị trí trùng xa nhất bên phải rồi dùng công thức

**Giải thích.** Gọi `p` là vị trí nhỏ nhất sao cho `nums[p:]` khác nhau. Một phần tử `nums[i]` buộc ta phải xóa nó nếu **cùng giá trị còn xuất hiện ở bên phải** (`nums[i] in nums[i + 1:]`). Vậy `p = 1 + max{ i : nums[i] xuất hiện lại bên phải }` (hoặc `0` nếu không có). Đáp án `ceil(p / 3) = (p + 2) // 3`. Cách này dùng toán trực tiếp, hợp để kiểm chứng Cách 3.

In [16]:
def min_ops_formula(nums: list[int]) -> int:
    p = 0
    for i in range(len(nums)):
        if nums[i] in nums[i + 1:]:          # giá trị này còn lặp lại ở bên phải
            p = i + 1                        # phải xóa tới hết vị trí i
    return (p + 2) // 3                      # ceil(p / 3)


for nums, expected in examples:
    print(min_ops_formula(nums), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 5:** `nums[i + 1:]` tạo bản sao và `in` quét ⇒ **Time `O(n²)`**, **Space `O(n)`** (với `n <= 100` không đáng kể). Muốn `O(n)` có thể dùng bảng chỉ số xuất hiện cuối cùng.

## 8. TRACE từng bước

Ta trace Cách 3 trên `nums = [1,2,3,4,2,3,3,5,7]` (đi từ phải sang trái):

| bước | `i` | `nums[i]` | `seen` trước | có trong `seen`? | hành động |
|---|---|---|---|---|---|
| 1 | 8 | 7 | `{}` | không | thêm 7 |
| 2 | 7 | 5 | `{7}` | không | thêm 5 |
| 3 | 6 | 3 | `{5,7}` | không | thêm 3 |
| 4 | 5 | 3 | `{3,5,7}` | **có** | trả về `5 // 3 + 1 = 2` |

Ý nghĩa: hậu tố `nums[6:] = [3,5,7]` khác nhau; thêm `nums[5] = 3` gây trùng với `3` đã có. Để loại phần tử này ta phải xóa **đến hết chỉ số `5`**, tức tiền tố dài `i + 1 = 6` phần tử. Hai thao tác xóa đúng `6` phần tử: `ceil(6 / 3) = 2` ✓.

In [17]:
def trace_min_ops(nums):
    seen = set()
    print("nums =", nums)
    for i in range(len(nums) - 1, -1, -1):
        x = nums[i]
        print(f"i={i}: nums[i]={x} | seen trước = {sorted(seen)} | có trong seen = {x in seen}")
        if x in seen:
            L = i + 1
            ans = (L + 2) // 3
            print(f"   -> phải xóa tiền tố dài {L}; ops = ceil({L}/3) = {ans}")
            return ans
        seen.add(x)
    print("   -> duyệt hết không trùng, ops = 0")
    return 0


trace_min_ops([1, 2, 3, 4, 2, 3, 3, 5, 7])
print()
trace_min_ops([4, 5, 6, 4, 4])

nums = [1, 2, 3, 4, 2, 3, 3, 5, 7]
i=8: nums[i]=7 | seen trước = [] | có trong seen = False
i=7: nums[i]=5 | seen trước = [7] | có trong seen = False
i=6: nums[i]=3 | seen trước = [5, 7] | có trong seen = False
i=5: nums[i]=3 | seen trước = [3, 5, 7] | có trong seen = True
   -> phải xóa tiền tố dài 6; ops = ceil(6/3) = 2

nums = [4, 5, 6, 4, 4]
i=4: nums[i]=4 | seen trước = [] | có trong seen = False
i=3: nums[i]=4 | seen trước = [4] | có trong seen = True
   -> phải xóa tiền tố dài 4; ops = ceil(4/3) = 2


2

## 9. Code nộp LeetCode

Ta chọn **Cách 3** (duyệt từ phải sang trái với `set`): ngắn, `O(n)` thời gian, dễ chứng minh.

In [18]:
class Solution:
    def minimumOperations(self, nums: list[int]) -> int:
        seen = set()
        for i in range(len(nums) - 1, -1, -1):
            if nums[i] in seen:               # nums[i:] có trùng, nums[i+1:] thì không
                return i // 3 + 1             # phải xóa tiền tố dài i + 1 => ceil((i + 1) / 3)
            seen.add(nums[i])
        return 0                              # đã khác nhau sẵn


for nums, expected in examples:
    print(Solution().minimumOperations(nums), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


## 10. Vì sao đúng

- **Sau `k` thao tác mảng là `nums[3k:]`.** Mỗi thao tác bỏ 3 phần tử đầu (hoặc hết). Vậy chỉ cần tìm `k` nhỏ nhất để `nums[3k:]` khác nhau.
- **Tính đơn điệu.** Nếu `nums[s:]` khác nhau thì `nums[s':]` với `s' >= s` cũng khác nhau (tập con của tập khác nhau vẫn khác nhau). Nên các `k` thỏa tạo thành một **khoảng tăng**, và `k` nhỏ nhất là ranh giới.
- **Vị trí ranh giới.** Duyệt từ phải sang trái, `seen` luôn chứa đúng các phần tử của `nums[i+1:]` và chúng khác nhau (vì chưa từng gặp trùng). Phần tử `nums[i]` đầu tiên có trong `seen` cho `nums[i:]` có trùng, còn `nums[i+1:]` khác nhau ⇒ vị trí bắt đầu nhỏ nhất của hậu tố khác nhau là `p = i + 1`.
- **Làm tròn lên.** Ta phải xóa **ít nhất** `p` phần tử đầu, mỗi thao tác xóa tối đa 3 ⇒ cần `ceil(p / 3)` thao tác, và `ceil(p/3)` thao tác xóa được `>= p` phần tử nên đủ. `ceil((i+1)/3) = (i+3)//3 = i//3 + 1`.
- **Không có trùng.** Duyệt hết không gặp trùng ⇒ mảng đã khác nhau ⇒ `0` thao tác.

## 11. Time/Space Complexity

Gọi `n = len(nums)`.

- **Time `O(n)`**: vòng lặp đi qua tối đa `n` phần tử; `in` và `add` của `set` trung bình `O(1)`.
- **Space `O(n)`**: `seen` chứa tối đa `n` giá trị (thực tế `<= 100` vì `nums[i] <= 100`, nên cũng có thể coi là `O(1)`).

Với `n <= 100` mọi cách đều chạy tức thì; cách `O(n)` là mẫu nên học vì mở rộng được cho `n` lớn (`10^5`).

## 12. Bộ test

### 12.1. Ví dụ của đề và gom các cách vào một bảng

In [19]:
impls = {
    "simulate": min_ops_simulate,
    "try_k": min_ops_try_k,
    "right_to_left": min_ops_right_to_left,
    "counter": min_ops_counter,
    "formula": min_ops_formula,
    "Solution": lambda nums: Solution().minimumOperations(nums),
}

for nums, expected in examples:
    for name, fn in impls.items():
        assert fn(nums) == expected, (name, nums)
print("PASS các ví dụ của đề với", len(impls), "cách")

PASS các ví dụ của đề với 6 cách


### 12.2. Edge case

- Mảng 1 phần tử, hai phần tử trùng nhau.
- Toàn một giá trị (`[7,7,7,7,7,7,7]`): cần xóa cho tới khi chỉ còn tối đa một phần tử.
- Đã khác nhau sẵn (đáp án `0`), kể cả độ dài `100` với các giá trị `1..100`.
- Trùng ngay ở hai phần tử cuối, hoặc ở hai phần tử đầu.
- Ranh giới chia hết cho 3 (`i + 1` bằng `3`, `6`) và không chia hết.

In [20]:
cases = [
    ([1], 0),
    ([5, 5], 1),                        # còn 2 phần tử, xóa hết bằng 1 thao tác
    ([7, 7, 7, 7, 7, 7, 7], 2),         # 7 phần tử: xóa 3 + 3, còn [7]
    ([1, 2, 3, 4, 5, 6], 0),
    ([1, 1, 2, 3, 4, 5], 1),            # chỉ cần xóa 3 đầu: [3,4,5]
    ([1, 2, 3, 4, 5, 1], 1),            # trùng ở i = 0 -> 0//3+1 = 1 ; xóa [1,2,3] còn [4,5,1]
    ([1, 2, 3, 3], 1),                  # i = 2 -> 2//3 + 1 = 1 ; xóa [1,2,3] còn [3]
    ([1, 2, 2, 3], 1),                  # i = 1 -> 1//3 + 1 = 1 ; còn [3]
    ([1, 1, 2, 3, 4, 5, 6, 6], 3),      # phản ví dụ của "trùng đầu tiên bên trái"
    ([3, 3, 3], 1),
    ([1, 2, 1, 2, 1, 2], 2),
]
for nums, expected in cases:
    ref = min_ops_simulate(nums)
    assert ref == expected, (nums, ref, expected)
    for name, fn in impls.items():
        assert fn(nums) == expected, (name, nums, fn(nums), expected)

full = list(range(1, 101))
for name, fn in impls.items():
    assert fn(full) == 0, name
print("PASS", len(cases), "edge case + mảng 1..100 x", len(impls), "cách")

PASS 11 edge case + mảng 1..100 x 6 cách


### 12.3. Phản ví dụ: duyệt trái sang phải và dừng ở trùng đầu tiên

Một cách nghĩ sai thường gặp: tìm phần tử đầu tiên (từ trái) bị trùng rồi xóa tới đó. Với `[1,1,2,3,4,5,6,6]` cách này cho kết quả sai, vì sau khi xóa `[1,1,2]` ta còn `[3,4,5,6,6]` vẫn trùng ở cuối.

In [21]:
def wrong_left_first_duplicate(nums):
    seen = set()
    for i, x in enumerate(nums):                   # SAI: dừng ở trùng đầu tiên từ bên trái
        if x in seen:
            return i // 3 + 1
        seen.add(x)
    return 0


nums = [1, 1, 2, 3, 4, 5, 6, 6]
print("sai:", wrong_left_first_duplicate(nums), "| đúng:", Solution().minimumOperations(nums))
assert wrong_left_first_duplicate(nums) != Solution().minimumOperations(nums)

sai: 1 | đúng: 3


### 12.4. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc

`n` từ `1` đến `100`, giá trị `1..100`. Để có nhiều trùng lặp ta thay đổi **kích thước miền giá trị** (`2, 5, 20, 100`). Đối chiếu tất cả các cách với **mô phỏng** (cách 1), và kiểm tra một tính chất: đáp án `<= ceil(n/3)`.

In [22]:
rng = random.Random(3396)
zero_count = 0
for trial in range(3000):
    n = rng.randint(1, 100)
    hi = rng.choice([2, 5, 20, 100])
    nums = [rng.randint(1, hi) for _ in range(n)]
    assert 1 <= len(nums) <= 100 and all(1 <= x <= 100 for x in nums)
    ref = min_ops_simulate(nums)
    assert 0 <= ref <= (n + 2) // 3
    zero_count += ref == 0
    for name, fn in impls.items():
        assert fn(nums) == ref, (name, nums)
print("PASS 3000 mảng ngẫu nhiên; số mảng có đáp án 0:", zero_count)

PASS 3000 mảng ngẫu nhiên; số mảng có đáp án 0: 171


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chia làm tròn xuống (`//`) mà quên làm tròn lên

In [23]:
def sai_lam_tron_xuong(nums):
    seen = set()
    for i in range(len(nums) - 1, -1, -1):
        if nums[i] in seen:
            return (i + 1) // 3                # SAI: tiền tố dài i + 1 cần ceil((i + 1) / 3)
        seen.add(nums[i])
    return 0


print("sai:", sai_lam_tron_xuong([4, 5, 6, 4, 4]), "| đúng:", Solution().minimumOperations([4, 5, 6, 4, 4]))
print("sai:", sai_lam_tron_xuong([5, 5]), "| đúng:", Solution().minimumOperations([5, 5]))

sai: 1 | đúng: 2
sai: 0 | đúng: 1


**Vì sao sai:** tiền tố cần xóa dài `4` (ở `[4,5,6,4,4]`), nhưng `4 // 3 = 1` chỉ xóa được `3` phần tử, còn `[4,4]` vẫn trùng. Làm tròn **lên** mới đủ: `(L + 2) // 3` hay `i // 3 + 1`. Với `[5,5]` (`L = 1`) công thức sai cho `0`, nhưng mảng chưa khác nhau nên đáp án đúng là `1`.

### Lỗi 2: Đếm số phần tử trùng thay vì vị trí trùng

In [24]:
def sai_dem_so_trung(nums):
    duplicates = len(nums) - len(set(nums))    # SAI: số phần tử "thừa"
    return (duplicates + 2) // 3


print("sai:", sai_dem_so_trung([4, 5, 6, 4, 4]), "| đúng:", Solution().minimumOperations([4, 5, 6, 4, 4]))
print("sai:", sai_dem_so_trung([1, 2, 3, 4, 5, 6, 7, 8, 9, 9]), "| đúng:", Solution().minimumOperations([1, 2, 3, 4, 5, 6, 7, 8, 9, 9]))

sai: 1 | đúng: 2
sai: 1 | đúng: 3


**Vì sao sai:** ta chỉ được xóa **từ đầu**, nên số ô bị xóa phụ thuộc **vị trí** của phần tử trùng, không phải **số lượng** phần tử trùng. Ở `[1,2,...,9,9]` chỉ có **một** giá trị thừa nhưng nó nằm ở cuối nên phải xóa tới tận `i = 8` (đáp án `3`), trong khi công thức sai chỉ cho `1`; ở `[4,5,6,4,4]` có hai giá trị thừa nhưng ta phải xóa tới `i = 3`. **Cách sửa:** tìm **vị trí** trùng xa nhất bên phải.

### Lỗi 3: Duyệt từ trái sang phải và dừng ở trùng đầu tiên

Đã minh họa ở mục 12.3: `[1,1,2,3,4,5,6,6]` cho kết quả sai vì sau khi xóa đến trùng đầu tiên vẫn còn trùng ở **bên phải**. **Cách sửa:** duyệt từ phải sang trái; trùng đầu tiên gặp được là trùng **xa nhất bên phải**, và xóa tới đó giải quyết mọi trùng ở bên trái nó.

### Lỗi 4: Mô phỏng nhưng quên cập nhật trạng thái (vòng lặp không tiến triển)

In [25]:
def sai_mo_phong_khong_tien(nums):
    arr = list(nums)
    ops = 0
    for _ in range(1000):                       # chốt chặn để ví dụ không treo máy
        if len(set(arr)) == len(arr):
            return ops
        arr = arr[:]                            # SAI: quên xóa 3 phần tử đầu -> không bao giờ tiến triển
        ops += 1
    return -1


print("sai (không tiến triển):", sai_mo_phong_khong_tien([4, 5, 6, 4, 4]), "| đúng:", Solution().minimumOperations([4, 5, 6, 4, 4]))

sai (không tiến triển): -1 | đúng: 2


**Vì sao sai:** mỗi vòng lặp **bắt buộc** làm mảng ngắn đi (`arr = arr[3:]`); nếu không, điều kiện dừng không bao giờ đổi và vòng `while` chạy vô hạn. **Cách sửa:** cập nhật trạng thái trong mỗi vòng, và để dễ kiểm tra hãy `print` độ dài mảng sau mỗi bước.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Chỉ được xóa từ đầu/cuối mảng**" ⇒ phần còn lại là **hậu tố/tiền tố**; thường chỉ cần tìm **ranh giới** của hậu tố tốt dài nhất.
- "**Các phần tử phải khác nhau**" ⇒ dùng `set`; kiểm tra bằng `len(set(a)) == len(a)`.
- Bài có tính **đơn điệu** (bớt phần tử không làm hỏng điều kiện) thường duyệt **từ phía không bị xóa** (ở đây là bên phải) và dừng ở lỗi đầu tiên gặp.
- Số thao tác để xóa `L` phần tử, mỗi lần tối đa `k`: `ceil(L / k) = (L + k - 1) // k`.
- Khi ràng buộc nhỏ (`n <= 100`), viết **mô phỏng** trước để làm "đáp án chuẩn" rồi tối ưu và đối chiếu.
- `nums[3:]` không bao giờ lỗi dù mảng ngắn; còn `nums[3]` thì có thể `IndexError`.

## 15. Biến thể của bài

1. **Xóa `k` phần tử đầu** mỗi lần (thay vì `3`): đáp án `(i + k) // k`.
2. **Xóa từ cuối** thay vì đầu ⇒ đối xứng, duyệt từ trái sang phải.
3. **Trả về chính mảng còn lại** (hoặc độ dài lớn nhất của hậu tố khác nhau).
4. **Cho phép xóa tùy ý** (không chỉ đầu mảng): số phần tử cần xóa `= n - số giá trị khác nhau` (đếm bằng `set`).
5. Bài liên quan: **Contains Duplicate (217)**, **Longest Substring Without Repeating Characters (3)**, **Minimum Operations to Make Array Distinct** (biến thể), **Remove Duplicates from Sorted Array (26)**.

In [26]:
def min_ops_k(nums, k):
    seen = set()
    for i in range(len(nums) - 1, -1, -1):
        if nums[i] in seen:
            return (i + 1 + k - 1) // k          # ceil((i + 1) / k)
        seen.add(nums[i])
    return 0


def longest_distinct_suffix(nums):
    seen = set()
    for i in range(len(nums) - 1, -1, -1):
        if nums[i] in seen:
            return nums[i + 1:]
        seen.add(nums[i])
    return nums[:]


def min_deletions_anywhere(nums):
    return len(nums) - len(set(nums))


print(min_ops_k([1, 2, 3, 4, 2, 3, 3, 5, 7], 3))        # 2 (trùng với đề)
print(min_ops_k([1, 2, 3, 4, 2, 3, 3, 5, 7], 2))        # 3
print(longest_distinct_suffix([1, 2, 3, 4, 2, 3, 3, 5, 7]))   # [3, 5, 7]
print(min_deletions_anywhere([4, 5, 6, 4, 4]))          # 2
assert min_ops_k([1, 2, 3, 4, 2, 3, 3, 5, 7], 3) == 2
assert min_ops_k([1, 2, 3, 4, 2, 3, 3, 5, 7], 2) == 3
assert longest_distinct_suffix([1, 2, 3, 4, 2, 3, 3, 5, 7]) == [3, 5, 7]
assert min_deletions_anywhere([4, 5, 6, 4, 4]) == 2

2
3
[3, 5, 7]
2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | xóa 3 đầu, dựng lại `set` mỗi vòng | `O(n²)` | `O(n)` | bám sát đề, làm đáp án chuẩn |
| 2. Thử từng `k` | kiểm tra hậu tố `nums[3k:]` | `O(n²)` | `O(n)` | dễ viết, hiểu cấu trúc hậu tố |
| 3. Phải sang trái + `set` | gặp trùng đầu tiên ⇒ `i // 3 + 1` | `O(n)` | `O(n)` | **nên nộp** |
| 4. `Counter` cập nhật | giảm tần suất khi xóa, theo dõi số giá trị trùng | `O(n)` | `O(n)` | mẫu "cửa sổ co lại bằng đếm" |
| 5. Công thức vị trí | `p = 1 + max i` có trùng bên phải | `O(n²)` | `O(n)` | kiểm chứng, giải thích toán |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[1,2,3,4,5][3:]`, `[1,2][3:]`, `len(set([1,1,2,2,3]))`, `(7 + 2) // 3`, `Solution().minimumOperations([1,2,3,3])`.

**Bài 2 (tính tay).** Với `nums = [5,5,1,2,3,4,4]`: lập bảng `k = 0, 1, 2, 3` với `nums[3k:]` và khác nhau hay không, rồi suy ra đáp án.

**Bài 3 (điền TODO).** Hoàn thành `min_ops_todo(nums)` theo Cách 3.

In [27]:
def min_ops_todo(nums):
    seen = set()
    # TODO 1: duyệt i từ len(nums) - 1 xuống 0 (dùng range(len(nums) - 1, -1, -1))
    #   TODO 2: nếu nums[i] đã có trong seen thì return i // 3 + 1
    #   TODO 3: ngược lại thêm nums[i] vào seen
    # TODO 4: nếu duyệt hết mà không trùng thì return 0
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `min_ops_last_index(nums)` dùng **dict ghi chỉ số xuất hiện cuối** (`last[x] = i`) hoặc `Counter` thay vì duyệt ngược: tìm `p = 1 + max{ i : nums[i] còn xuất hiện ở bên phải }` rồi trả về `(p + 2) // 3` (gợi ý: giá trị `x` ở chỉ số `i` "còn xuất hiện bên phải" nếu `last[x] > i`).

**Bài 5 (biến thể / mini challenge).** (a) Viết `min_ops_with_k(nums, k)` cho thao tác xóa `k` phần tử đầu (đáp án `(i + k) // k` khi gặp trùng ở `i`). (b) Viết `remaining_after_ops(nums)` trả về **mảng còn lại** sau khi thực hiện số thao tác tối thiểu.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[4, 5]`, `[]`, `3`, `3`, `1`. (Với `[1,2,3,3]`: duyệt ngược `3` rồi `3` trùng ở `i = 2`, đáp án `2 // 3 + 1 = 1`.) Bài 3: dùng vòng `for` với `range` đếm lùi và `set`.
- **Mức 2 (vừa):** Bài 2: `k=0`: `[5,5,1,2,3,4,4]` trùng; `k=1`: `[2,3,4,4]` trùng; `k=2`: `[4]` khác nhau ⇒ đáp án `2`. Bài 4: tạo `last = {x: i for i, x in enumerate(nums)}` (ghi đè nên giữ chỉ số cuối).
- **Mức 3 (sâu):** Bài 5(a): thay `3` bằng `k` trong `i // k + 1`. 5(b): sau khi có `ops`, mảng còn lại là `nums[3 * ops:]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert [1, 2, 3, 4, 5][3:] == [4, 5]
assert [1, 2][3:] == []
assert len(set([1, 1, 2, 2, 3])) == 3
assert (7 + 2) // 3 == 3
assert Solution().minimumOperations([1, 2, 3, 3]) == 1

# Bài 2
nums2 = [5, 5, 1, 2, 3, 4, 4]
assert [len(set(nums2[3 * k:])) == len(nums2[3 * k:]) for k in range(4)] == [False, False, True, True]
assert Solution().minimumOperations(nums2) == 2


# Bài 3
def min_ops_todo_ans(nums):
    seen = set()
    for i in range(len(nums) - 1, -1, -1):
        if nums[i] in seen:
            return i // 3 + 1
        seen.add(nums[i])
    return 0


for nums, expected in examples:
    assert min_ops_todo_ans(nums) == expected


# Bài 4
def min_ops_last_index(nums):
    last = {x: i for i, x in enumerate(nums)}          # chỉ số xuất hiện cuối của mỗi giá trị
    p = 0
    for i, x in enumerate(nums):
        if last[x] > i:                                 # x còn xuất hiện ở bên phải
            p = i + 1
    return (p + 2) // 3


for nums, expected in examples:
    assert min_ops_last_index(nums) == expected
assert min_ops_last_index(nums2) == 2


# Bài 5
def min_ops_with_k(nums, k):
    seen = set()
    for i in range(len(nums) - 1, -1, -1):
        if nums[i] in seen:
            return i // k + 1
        seen.add(nums[i])
    return 0


def remaining_after_ops(nums):
    ops = Solution().minimumOperations(nums)
    return nums[3 * ops:]


assert min_ops_with_k([1, 2, 3, 4, 2, 3, 3, 5, 7], 3) == 2
assert min_ops_with_k([1, 2, 3, 4, 2, 3, 3, 5, 7], 2) == 3
assert remaining_after_ops([1, 2, 3, 4, 2, 3, 3, 5, 7]) == [3, 5, 7]
assert remaining_after_ops([4, 5, 6, 4, 4]) == []
assert remaining_after_ops([6, 7, 8, 9]) == [6, 7, 8, 9]
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (xóa tiền tố, tìm ranh giới hậu tố không trùng bằng `set` từ phải sang trái)

```python
class Solution:
    def minimumOperations(self, nums: List[int]) -> int:
        seen = set()
        for i in range(len(nums) - 1, -1, -1):
            if nums[i] in seen:
                return i // 3 + 1
            seen.add(nums[i])
        return 0
```

**Checklist**

- [ ] Nhớ rằng sau `k` thao tác còn lại `nums[3k:]` (chỉ xóa tiền tố)?
- [ ] Duyệt **từ phải sang trái** và dừng ở trùng **đầu tiên gặp** (xa nhất bên phải)?
- [ ] Làm tròn **lên**: `i // 3 + 1` hoặc `(L + 2) // 3` với `L = i + 1`?
- [ ] Xử lý mảng đã khác nhau sẵn (trả `0`) và còn ít hơn 3 phần tử?
- [ ] Đã thử `[4,5,6,4,4]`, `[5,5]`, và phản ví dụ `[1,1,2,3,4,5,6,6]` (đáp án `3`)?
- [ ] Skeleton đúng `class Solution` / `def minimumOperations(self, nums: List[int]) -> int`?